In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
import validators
import mlflow
from mlflow.tracking import MlflowClient
from scipy import stats

In [ ]:
MLFLOW_TRACKING_URI = "http://localhost:2000"
MLFLOW_EXPERIMENT_NAME = "compare_datasets"
MLFLOW_RUN_NAME = "frist run"

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

In [ ]:
local_path = client.download_artifacts(run_id="f8de7388f50b47eea028f893fbc6ebf9", path="data/lit_author_metrics.csv")

ruslit_metrics = pd.read_csv(local_path)
ruslit_metrics.head()

In [ ]:
local_path = client.download_artifacts(run_id="21969b89951c44b8885f35c231b29a94", path="data/socials_author_metrics.csv")

social_metrics = pd.read_csv(local_path)
social_metrics.head()

In [ ]:
local_path = client.download_artifacts(run_id="01712f9f3c37472a963f1042d6402506", path="data/npl1_author_metrics.csv")

nplus1_metrics = pd.read_csv(local_path)
nplus1_metrics.head()

In [ ]:
nltk.download('stopwords')
ru_stopwords = set(nltk.corpus.stopwords.words("russian"))

In [ ]:
re_id = re.compile(r'id\d+')
def clean_text(text: str) -> str | None:
    splited = text.split()
    result = []

    for item in splited:
        if 'id' in text:
            pass
        if re_id.search(item):
            continue
        if validators.url(item):
            continue

        _item = item.replace('\xa0', '')
        _item = _item.replace('\\n', '')
        result.append(_item)

    if not result:
        return None

    return ' '.join(result)

In [ ]:
socials_df = pd.read_csv('../data_/socials.csv')
socials_df.columns=["author","text"]
socials_df.head()

In [ ]:
ruslit_df = pd.read_csv('../data_/ruslit.csv', delimiter=',')
ruslit_df.columns=["author","title", 'text']
ruslit_df.head()

In [ ]:
ruslit_df = pd.read_csv('../../data/data/balanced_lit.csv', delimiter=',')
ruslit_df.head()

In [ ]:
socials_df = pd.read_csv('../../data/data/socials.csv', header=None)
socials_df.columns = ["author","text"]
socials_df.head()

In [ ]:
nplus1_df = pd.read_csv('../../data/data/npl1.csv', delimiter=',')
nplus1_df.head()

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
mlflow.start_run(run_name=MLFLOW_RUN_NAME)

In [ ]:
mlflow.set_tag("upstream.ruslit_analysis_run_id", 'f8de7388f50b47eea028f893fbc6ebf9')
mlflow.set_tag("upstream.socials_analysis_run_id", '21969b89951c44b8885f35c231b29a94')
mlflow.set_tag("upstream.nplus1_analysis_run_id", '01712f9f3c37472a963f1042d6402506')

In [ ]:

lens = [
    {
        'df': 'socials_df',
        'len':len(socials_df),
    },
    {
        'df': 'ruslit_df',
        'len':len(ruslit_df),
    },
    {
        'df': 'nplus1_df',
        'len':len(nplus1_df),
    }
]
lens = pd.DataFrame(lens)

plt.figure(figsize=(8, 6))
ax = sns.barplot(x='df', y='len',data=lens)

ax.bar_label(ax.containers[0])
plt.show()

In [ ]:
socials_df['len'] = [len(x) for x in socials_df['text']]
ruslit_df['len'] = [len(x) for x in ruslit_df['text']]
nplus1_df['len'] = [len(x) for x in nplus1_df['text']]

mean_text_lens = [
    {
        'df': 'socials_df',
        'len': socials_df['len'].mean(),
    },
    {
        'df': 'ruslit_df',
        'len': ruslit_df['len'].mean(),
    },
    {
        'df': 'nplus1_df',
        'len': nplus1_df['len'].mean(),
    }
]
mean_text_lens = pd.DataFrame(mean_text_lens)

plt.figure(figsize=(8, 6))
ax = sns.barplot(x='df', y='len',data=mean_text_lens)
ax.bar_label(ax.containers[0])
plt.show()

In [ ]:
socials_author_grouped = socials_df['author'].value_counts()

ruslit_author_grouped = ruslit_df['author'].value_counts()

nplus1_df_author_grouped = nplus1_df['author'].value_counts()

In [ ]:
grouped_lens = [
    {
        'df': 'socials_df',
        'len':len(socials_author_grouped),
    },
    {
        'df': 'ruslit_df',
        'len':len(ruslit_author_grouped),
    },
    {
        'df': 'nplus1_df',
        'len':len(nplus1_df_author_grouped),
    }
]
grouped_lens = pd.DataFrame(grouped_lens)

plt.figure(figsize=(8, 6))
ax = sns.barplot(x='df', y='len',data=grouped_lens)

ax.bar_label(ax.containers[0])

plt.show()

In [ ]:
socials_df["text"] = socials_df["text"].apply(
    lambda x:
    clean_text(x)
)
# ruslit_df["text"] = ruslit_df["text"].apply(
#     lambda x:
#     clean_text(x)
# )
nplus1_df["text"] = nplus1_df["text"].apply(
    lambda x:
    clean_text(x)
)

## Анализ метрик
### Метрики для литературы

In [ ]:
def is_data_normal(df, metric_name: str, a: float = 0.05) -> bool:
    _, p_normal_s = stats.shapiro(df[metric_name].dropna())
    if p_normal_s > 0.05:
        print(f"Данные по метрике {metric_name} нормально распределены ({p_normal_s})")
    else:
        print(f"Данные по метрике {metric_name} НЕ нормально распределены ({p_normal_s})")

In [ ]:
metrics = [
    'avg_sent_len_words',
    'avg_word_len_chars',

    'mtld',

    'avg_tree_depth',
    'passive_ratio',
    'func_word_ratio',
    'avg_clauses_per_sent',
    'noun_verb_ratio',

    'compression_ratio',

    'flesch_reading_ease',
    'gunning_fog',
]

In [ ]:
socials_metrics_original = social_metrics.copy()

In [ ]:
percentile = 99.9
filter_metrics = [
    'avg_sent_len_words',
    'avg_word_len_chars',
    'mtld',
    'flesch_reading_ease',
    'gunning_fog',
]
social_metrics_filtered = social_metrics.copy()
for metric in filter_metrics:
    before_metric_copy = social_metrics.copy()
    for suf in ['_median', '_mean']:

        threshold = social_metrics[metric + suf].quantile(percentile / 100)
        mask = social_metrics[metric + suf] <= threshold

        #social_metrics = social_metrics[(social_metrics[metric + suf] >= lower_bound) & (social_metrics[metric + suf] <= upper_bound)]
        social_metrics_filtered = social_metrics_filtered[mask]

        removed_rows = before_metric_copy[~before_metric_copy.index.isin(social_metrics_filtered.index)]

        print('Metric: ', metric + suf)
        print("Removed rows:", len(removed_rows))

        filtered_authors = removed_rows['author'].unique()

        print(f"Number of rows filtered: {len(removed_rows)}")
        print(f"Number of unique authors filtered: {len(filtered_authors)}")
        print(f"\nFiltered authors:\n{filtered_authors}")

        for author in filtered_authors:
            author_texts = socials_df[socials_df['author'] == author]['text'].head(3)
            print(f"\n--- Author: {author} ---")
            print(f"Number of texts: {len(socials_df[socials_df['author'] == author])}")
            for idx, text in enumerate(author_texts, 1):
                print(f"Example {idx}: {text[:200]}...")  # First 200 chars


In [ ]:
len(social_metrics) - len(social_metrics_filtered)

In [ ]:
def show_kde(metric: str, filtered=False):
    fig, ax = plt.subplots(figsize=(12, 5))

    dif_dict = {}

    sns.kdeplot(ruslit_metrics[metric], label='Литература', ax=ax)
    sns.kdeplot(social_metrics[metric], label='Комментарии из соцсетей', ax=ax)
    sns.kdeplot(nplus1_metrics[metric], label='nplus1', ax=ax)

    ax.set_title(f"Распределение {metric} {'Выбросы удалены' if filtered else 'Без удаления выбросов'}")

    _, p_normal_ruslit = stats.shapiro(ruslit_metrics[metric].dropna())
    _, p_normal_scinews = stats.shapiro(nplus1_metrics[metric].dropna())
    _, p_normal_socials = stats.shapiro(social_metrics[metric].dropna())

    dif_dict['ruslit_nplus'] = {}

    if p_normal_ruslit > 0.05 and p_normal_scinews > 0.05:
        stat, p = stats.ttest_ind(
            ruslit_metrics[metric].dropna(),
            nplus1_metrics[metric].dropna()
        )
        dif_dict['ruslit_nplus']['test'] = 't-test'
    else:
        stat, p = stats.mannwhitneyu(
            ruslit_metrics[metric].dropna(),
            nplus1_metrics[metric].dropna()
        )
        dif_dict['ruslit_nplus']['test'] = 'mann-whitneyu'

    print(
        f"Различие между РусЛитрой и НаучЖурналом по метрике {metric} значимо {p}" if p < 0.05
        else f"Различия между РусЛитрой и НаучЖурналом по метрике {metric} нет {p}"
    )

    dif_dict['ruslit_nplus']['p'] = p

    dif_dict['ruslit_socials'] = {}
    if p_normal_ruslit > 0.05 and p_normal_socials > 0.05:
        stat, p = stats.ttest_ind(
            ruslit_metrics[metric].dropna(),
            social_metrics[metric].dropna()
        )
        dif_dict['ruslit_socials']['test'] = 't-test'
    else:
        stat, p = stats.mannwhitneyu(
            ruslit_metrics[metric].dropna(),
            social_metrics[metric].dropna()
        )
        dif_dict['ruslit_socials']['test'] = 'mann-whitneyu'
    print(
            f"Различие между РусЛитрой и СоцСетями по метрике {metric} значимо {p}" if p < 0.05
            else f"Различия между РусЛитрой и СоцСетями по метрике {metric} нет {p}"
        )

    dif_dict['ruslit_socials']['p'] = p

    dif_dict['nplus_socials'] = {}
    if p_normal_socials > 0.05 and p_normal_scinews > 0.05:
        stat, p = stats.ttest_ind(
            nplus1_metrics[metric].dropna(),
            social_metrics[metric].dropna()
        )

        dif_dict['nplus_socials']['test'] = 't-test'
    else:
        stat, p = stats.mannwhitneyu(
            nplus1_metrics[metric].dropna(),
            social_metrics[metric].dropna()
        )
        dif_dict['nplus_socials']['test'] = 'mann-whitneyu'

    print(
        f"Различие между СоцСетями и НаучЖурналом по метрике {metric} значимо {p}" if p < 0.05
        else f"Различия между СоцСетями и НаучЖурналом по метрике {metric} нет {p}"
    )
    dif_dict['nplus_socials']['p'] = p

    p_text = (
        f"Литература vs Комментарии соц-сетей: p={dif_dict['ruslit_socials']['p']} ({dif_dict['ruslit_socials']['test']})\n"
        f"Литература vs Nplus1: p={dif_dict['ruslit_nplus']['p']} ({dif_dict['ruslit_nplus']['test']})\n"
        f"Комментарии соц-сетей vs Nplus1: p={dif_dict['nplus_socials']['p']} ({dif_dict['nplus_socials']['test']})"
    )

    ax.text(
        0.98, 0.97, p_text,
        transform=ax.transAxes,
        fontsize=9, verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )

    ax.legend()
    fig.tight_layout()
    display(fig)

    mlflow.log_figure(fig, f"plots/kde_{metric}{'_filtered' if filtered else ''}.png")
    plt.close(fig)

    return dif_dict

In [ ]:
result_dict = {}
for metric in metrics:
    dif_dict_mean = show_kde(metric + '_mean')
    dif_dict_median = show_kde(metric + '_median')
    result_dict[metric + '_mean'] = dif_dict_mean
    result_dict[metric + '_median'] = dif_dict_median

mlflow.log_dict(result_dict, 'data/kde_results.json')

In [ ]:
social_metrics = social_metrics_filtered.copy()
result_dict = {}
for metric in metrics:
    dif_dict_mean = show_kde(metric + '_mean', filtered=True)
    dif_dict_median = show_kde(metric + '_median', filtered=True)
    result_dict[metric + '_mean'] = dif_dict_mean
    result_dict[metric + '_median'] = dif_dict_median

mlflow.log_dict(result_dict, 'data/kde_filtered_results.json')

In [ ]:
mlflow.end_run()